Sources

In [ ]:
#Video source - https://www.kaggle.com/datasets/hmai11/crema-d
#Vector database - https://app.pinecone.io/organizations/

Import libraries

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
import random
from pathlib import Path
import subprocess

Get video repository

In [ ]:
video_dir='./Videos/VideoFlash/'
video_ext='*.flv'
all_videos=[]
def getFiles(filePath, fileExt, storeObj):
    for path in Path(filePath).rglob(fileExt):
        if path.is_file():
            storeObj.append(path.name)

getFiles(video_dir, video_ext, all_videos)
random_videos = random.sample(all_videos,1000)

len(random_videos)

Convert video to audio

In [4]:
def convertVideoToAudio(filePath, index):
    command = "ffmpeg -i "+ filePath +" -vn -q:a 0 -map a ./Converted_Audio/output-audio_"+str(index)+".mp3"
    subprocess.run(command, shell=True)

counter = 1
for video in random_videos:
    videoPath = './Videos/VideoFlash/'+ video
    convertVideoToAudio(videoPath, counter)
    counter = counter + 1

In [ ]:
audio_dir = './Converted_Audio/'
audio_ext = '*.mp3'
all_audios=[]
getFiles(audio_dir, audio_ext, all_audios)
random_audios = random.sample(all_audios, 100)
len(random_audios)


Setup env variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../var.env')
vect_api_key = os.getenv("VECT_API_KEY")
azure_api_key = os.getenv("AZURE_API_KEY")
az_endpoint = os.getenv("AZURE_ENDPOINT")
az_ver = os.getenv("AZURE_VERSION")
deploy_model = os.getenv("DEPLOYMENT_MODEL")
semantic_model = os.getenv("SEMANTIC_MODEL")
#print(azure_api_key) - Test if your env varaibles are loading or not.

Azure open AI setup

In [30]:

from openai import AzureOpenAI
client = AzureOpenAI(
        api_key=azure_api_key,
        azure_endpoint=az_endpoint,
        api_version=az_ver
    )

data=[]

Extract text using AI audio transcribe from audio files

In [31]:
def getTextFromAudio(file):
    transcript = client.audio.transcriptions.create(
                    model=deploy_model,
                    file=file
                )
    return transcript.text

Create semantic model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(semantic_model, device='cpu')

Create dataset for audio file name and it's text and embeddings

In [ ]:
import pandas as pd

for i in tqdm(range(0, len(random_audios))):
    filePath = audio_dir + random_audios[i]
    audio_file = open(filePath, "rb")
    audio_text = getTextFromAudio(audio_file)
    embeddings = model.encode(audio_text)
    data.append({'Name':random_audios[i], 'AudioText':audio_text, 'Values':embeddings})

df = pd.DataFrame(data)

df['metadata']=df[['Name','AudioText']].to_dict(orient='records')
df['id']=df.reset_index(drop='index').index

Pinecone init

In [ ]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=vect_api_key)
pc.create_index(name='semantic-audio-text', metric='cosine', spec=ServerlessSpec(cloud='aws', region='us-east-1'), dimension=384)

Data insert into vector DB

In [ ]:
df['id']=df['id'].astype('str')
df_upsert = df[['id', 'metadata', 'values']]
df_upsert.unique()
vectIdx = pc.Index(name='semantic-audio-text')
vectIdx.upsert_from_dataframe(df_upsert)

Query vector database

In [ ]:
vectIdx.query(vector=(model.encode("I'll wear a jacket for a meeting today at eleven")).tolist(), top_k=5, include_metadata=True)